<a href="https://colab.research.google.com/github/duttaprat/BMI_503/blob/main/class_2/GENCODE_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Genomic Annotations GENCODE



**Course**: BMI 503 - Introduction to Computer Science for Biomedical Informatics  
**Instructors**: Pratik Dutta   
**Institution**: Stony Brook University

---



## 📚 Table of Contents

1. [Introduction to GENCODE](#section1)
2. [Understanding GTF and GFF Formats](#section2)
3. [Downloading and Parsing GENCODE Data](#section3)
4. [Creating a DataFrame from GTF](#section4)
5. [Extracting TSS (Transcription Start Sites)](#section5)
6. [Finding Splice Sites](#section6)
7. [Extracting Promoter Regions (-1kb to TSS)](#section7)
8. [Getting FASTA Sequences](#section8)
9. [Regular Expressions and Motif Finding](#section9)



## Setup & Installation

In [4]:
# Install required packages
!pip install pandas biopython requests -q

import pandas as pd
import re
from collections import defaultdict
import matplotlib.pyplot as plt
import seaborn as sns

print("✅ Setup complete!")

✅ Setup complete!


##  Downloading and Parsing GENCODE Data

### Where to Get GENCODE Data?

**GENCODE Website:** https://www.gencodegenes.org/

In [5]:
import urllib.request
import gzip


# Full genome GTF is ~1.5 GB!
# URL for human GRCh38 release 49
url = "https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_49/gencode.v49.annotation.gtf.gz"

print("📥 Downloading GENCODE GTF...")
output_file = "gencode.v49.annotation.gtf.gz"
urllib.request.urlretrieve(url, output_file)
print(f"✅ Downloaded: {output_file}")

📥 Downloading GENCODE GTF...
✅ Downloaded: gencode.v49.annotation.gtf.gz


In [6]:
# Parse only chromosome 22
print("\n📖 Parsing GTF file (chr22 only)...")

data = []
target_chr = 'chr22'  # Change to 'chr1', 'chr17', etc.

with gzip.open(output_file, 'rt') as f:
    for line in f:
        if line.startswith('#'):
            continue

        fields = line.strip().split('\t')
        if len(fields) < 9:
            continue

        # Only process target chromosome
        if fields[0] != target_chr:
            continue

        record = {
            'seqname': fields[0],
            'source': fields[1],
            'feature': fields[2],
            'start': int(fields[3]),
            'end': int(fields[4]),
            'score': fields[5],
            'strand': fields[6],
            'frame': fields[7],
        }

        # Parse attributes
        attributes = fields[8]
        for match in [
            ('gene_id', r'gene_id "([^"]+)"'),
            ('gene_name', r'gene_name "([^"]+)"'),
            ('transcript_id', r'transcript_id "([^"]+)"'),
            ('gene_type', r'gene_type "([^"]+)"')
        ]:
            m = re.search(match[1], attributes)
            if m:
                record[match[0]] = m.group(1)

        data.append(record)

df = pd.DataFrame(data)
print("✅ GTF parsed into DataFrame!")
print(f"\nShape: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"\nColumns: {list(df.columns)}")
df


📖 Parsing GTF file (chr22 only)...
✅ GTF parsed into DataFrame!

Shape: 191484 rows × 12 columns

Columns: ['seqname', 'source', 'feature', 'start', 'end', 'score', 'strand', 'frame', 'gene_id', 'gene_name', 'gene_type', 'transcript_id']


,seqname,source,feature,start,end,score,strand,frame,gene_id,gene_name,gene_type,transcript_id
0,chr22,HAVANA,gene,10524345,10529164,.,-,.,ENSG00000294541.1,ENSG00000294541,lncRNA,NaN
1,chr22,HAVANA,transcript,10524345,10529164,.,-,.,ENSG00000294541.1,ENSG00000294541,lncRNA,ENST00000724296.1
2,chr22,HAVANA,exon,10529036,10529164,.,-,.,ENSG00000294541.1,ENSG00000294541,lncRNA,ENST00000724296.1
3,chr22,HAVANA,exon,10527853,10528040,.,-,.,ENSG00000294541.1,ENSG00000294541,lncRNA,ENST00000724296.1
4,chr22,HAVANA,exon,10524345,10524446,.,-,.,ENSG00000294541.1,ENSG00000294541,lncRNA,ENST00000724296.1
...,...,...,...,...,...,...,...,...,...,...,...,...
191479,chr22,HAVANA,exon,50798788,50798991,.,-,.,ENSG00000307555.1,ENSG00000307555,lncRNA,ENST00000827060.1
191480,chr22,HAVANA,exon,50797082,50797504,.,-,.,ENSG00000307555.1,ENSG00000307555,lncRNA,ENST00000827060.1
191481,chr22,HAVANA,gene,50798655,50799123,.,+,.,ENSG00000184319.19,RPL23AP82,transcribed_unprocessed_pseudogene,NaN
191482,chr22,HAVANA,transcript,50798655,50799123,.,+,.,ENSG00000184319.19,RPL23AP82,transcribed_unprocessed_pseudogene,ENST00000427528.1


In [11]:
print(f"✅ Parsed {target_chr}!")
print(f"   Features: {len(df):,}")
print(f"   Genes: {df[df['feature'] == 'gene'].shape[0]:,}")

✅ Parsed chr22!
   Features: 191,484
   Genes: 1,747


In [12]:
# Let's explore our parsed data

print("="*70)
print("📊 EXPLORING THE GENCODE DATA")
print("="*70)

print(f"\n1. Feature types:")
print(df['feature'].value_counts())

print(f"\n2. Genes in our sample:")
genes = df[df['feature'] == 'gene']
display(genes[['seqname', 'gene_name', 'start', 'end', 'strand']])



📊 EXPLORING THE GENCODE DATA

1. Feature types:
feature
exon              90130
CDS               58986
UTR               18226
transcript        11615
start_codon        5491
stop_codon         5277
gene               1747
Selenocysteine       12
Name: count, dtype: int64

2. Genes in our sample:


,seqname,gene_name,start,end,strand
0,chr22,ENSG00000294541,10524345,10529164,-
5,chr22,U2,10736171,10736283,-
8,chr22,ENSG00000301473,10742050,10753085,+
16,chr22,ENSG00000301496,10752636,10754413,-
20,chr22,FRG1FP,10939388,10961338,-
...,...,...,...,...,...
188673,chr22,SNRPA1P2,50754675,50755434,-
188676,chr22,ENSG00000291064,50756831,50801309,+
189215,chr22,RABL2B,50767491,50783760,-
191477,chr22,ENSG00000307555,50797082,50798991,-


In [13]:
print(f"\n3. Transcripts per gene:")
transcripts = df[df['feature'] == 'transcript']
print(transcripts.groupby('gene_name').size())

print(f"\n4. Exons per transcript:")
exons = df[df['feature'] == 'exon']
if 'transcript_id' in exons.columns:
    print(exons.groupby('transcript_id').size())


3. Transcripts per gene:
gene_name
5_8S_rRNA     1
A4GALT       55
ABCD1P4       1
ABHD17AP4     1
ABHD17AP5     1
             ..
ZNF73P        1
ZNF74        11
ZNRF3         9
ZNRF3-AS1     4
ZNRF3-IT1     1
Length: 1705, dtype: int64

4. Exons per transcript:
transcript_id
ENST00000006251.11     9
ENST00000008876.7     10
ENST00000043402.8      2
ENST00000086933.3      3
ENST00000155674.9      8
                      ..
ENST00000972527.1     19
ENST00000972528.1     19
ENST00000972529.1     19
ENST00000972530.1     19
ENST00000972531.1      7
Length: 11615, dtype: int64


In [14]:
# Create separate DataFrames for different features

genes_df = df[df['feature'] == 'gene'].copy()
transcripts_df = df[df['feature'] == 'transcript'].copy()
exons_df = df[df['feature'] == 'exon'].copy()
cds_df = df[df['feature'] == 'CDS'].copy()

print("✅ Created feature-specific DataFrames:")
print(f"   Genes:       {len(genes_df)}")
print(f"   Transcripts: {len(transcripts_df)}")
print(f"   Exons:       {len(exons_df)}")
print(f"   CDS:         {len(cds_df)}")

✅ Created feature-specific DataFrames:
   Genes:       1747
   Transcripts: 11615
   Exons:       90130
   CDS:         58986


##  Extracting TSS (Transcription Start Sites)

### What is TSS?

```
TSS = Transcription Start Site
     = Where RNA polymerase begins transcribing

For + strand genes: TSS = start position
For - strand genes: TSS = end position

Example:

+ strand:  ======gene======>
           ^
           TSS (at start)

- strand:  <======gene======
                           ^
                           TSS (at end)
```

Let's dive into the code! 👇

In [15]:
### Extract TSS from Transcripts


def extract_tss(transcripts_df):
    """
    Extract TSS from transcript annotations

    TSS depends on strand:
    - Plus strand (+): TSS is at start position
    - Minus strand (-): TSS is at end position
    """

    tss_data = []

    for idx, row in transcripts_df.iterrows():
        # Calculate TSS based on strand
        if row['strand'] == '+':
            tss = row['start']
        else:  # minus strand
            tss = row['end']

        tss_data.append({
            'gene_id': row.get('gene_id', ''),
            'gene_name': row.get('gene_name', ''),
            'transcript_id': row.get('transcript_id', ''),
            'chromosome': row['seqname'],
            'tss': tss,
            'strand': row['strand'],
            'gene_start': row['start'],
            'gene_end': row['end']
        })

    return pd.DataFrame(tss_data)

# Extract TSS
tss_df = extract_tss(transcripts_df)

print("✅ Extracted TSS for all transcripts!")
display(tss_df)
print(f"\nTotal TSS: {len(tss_df)}")



✅ Extracted TSS for all transcripts!


,gene_id,gene_name,transcript_id,chromosome,tss,strand,gene_start,gene_end
0,ENSG00000294541.1,ENSG00000294541,ENST00000724296.1,chr22,10529164,-,10524345,10529164
1,ENSG00000277248.1,U2,ENST00000615943.1,chr22,10736283,-,10736171,10736283
2,ENSG00000301473.1,ENSG00000301473,ENST00000779064.1,chr22,10742050,+,10742050,10753085
3,ENSG00000301473.1,ENSG00000301473,ENST00000779065.1,chr22,10742092,+,10742092,10753081
4,ENSG00000301496.1,ENSG00000301496,ENST00000779294.1,chr22,10754413,-,10752636,10754413
...,...,...,...,...,...,...,...,...
11610,ENSG00000079974.20,RABL2B,ENST00000468451.1,chr22,50783600,-,50775771,50783600
11611,ENSG00000079974.20,RABL2B,ENST00000464740.1,chr22,50783630,-,50777660,50783630
11612,ENSG00000079974.20,RABL2B,ENST00000413505.1,chr22,50783045,-,50782234,50783045
11613,ENSG00000307555.1,ENSG00000307555,ENST00000827060.1,chr22,50798991,-,50797082,50798991



Total TSS: 11615


## Finding Splice Sites

### What are Splice Sites?

```
Splice sites = Junctions between exons and introns

Pre-mRNA:  exon1 - intron - exon2 - intron - exon3
                  ^        ^        ^        ^
                  splice sites

Donor site (5'): End of exon (usually GT)
Acceptor site (3'): Start of exon (usually AG)
```

In [17]:
### Extract Splice Sites


def extract_splice_sites(exons_df):
    """
    Extract splice sites from exon annotations

    For each transcript:
    - Find junctions between consecutive exons
    - Donor = end of upstream exon
    - Acceptor = start of downstream exon
    """

    splice_sites = []

    # Group by transcript
    for transcript_id, group in exons_df.groupby('transcript_id'):
        # Sort exons by position
        group = group.sort_values('start')

        # Get strand
        strand = group.iloc[0]['strand']
        chrom = group.iloc[0]['seqname']
        gene_name = group.iloc[0].get('gene_name', '')

        # Iterate through consecutive exons
        exon_list = group.sort_values('exon_number' if 'exon_number' in group.columns else 'start')

        for i in range(len(exon_list) - 1):
            exon1 = exon_list.iloc[i]
            exon2 = exon_list.iloc[i + 1]

            if strand == '+':
                donor = exon1['end']
                acceptor = exon2['start']
            else:
                donor = exon2['start']
                acceptor = exon1['end']

            splice_sites.append({
                'transcript_id': transcript_id,
                'gene_name': gene_name,
                'chromosome': chrom,
                'strand': strand,
                'donor_site': donor,
                'acceptor_site': acceptor,
                'intron_start': min(donor, acceptor),
                'intron_end': max(donor, acceptor),
                'intron_length': abs(acceptor - donor)
            })

    return pd.DataFrame(splice_sites)

# Extract splice sites
if len(exons_df) > 0 and 'transcript_id' in exons_df.columns:
    splice_sites_df = extract_splice_sites(exons_df)

    print("✅ Extracted splice sites!")
    print(f"\nTotal splice junctions: {len(splice_sites_df)}")
    print(f"\nSplice sites data:")
    display(splice_sites_df)

    print(f"\n📊 Intron length statistics:")
    print(f"   Mean: {splice_sites_df['intron_length'].mean():.0f} bp")
    print(f"   Min:  {splice_sites_df['intron_length'].min()} bp")
    print(f"   Max:  {splice_sites_df['intron_length'].max()} bp")
else:
    print("⚠️ Not enough exon data for splice site extraction")

✅ Extracted splice sites!

Total splice junctions: 78515

Splice sites data:


,transcript_id,gene_name,chromosome,strand,donor_site,acceptor_site,intron_start,intron_end,intron_length
0,ENST00000006251.11,PRR5,chr22,+,44677240,44702492,44677240,44702492,25252
1,ENST00000006251.11,PRR5,chr22,+,44702608,44714591,44702608,44714591,11983
2,ENST00000006251.11,PRR5,chr22,+,44714671,44725244,44714671,44725244,10573
3,ENST00000006251.11,PRR5,chr22,+,44725292,44726577,44725292,44726577,1285
4,ENST00000006251.11,PRR5,chr22,+,44726634,44731730,44726634,44731730,5096
...,...,...,...,...,...,...,...,...,...
78510,ENST00000972531.1,PISD,chr22,-,31620996,31620599,31620599,31620996,397
78511,ENST00000972531.1,PISD,chr22,-,31621334,31621142,31621142,31621334,192
78512,ENST00000972531.1,PISD,chr22,-,31648101,31621472,31621472,31648101,26629
78513,ENST00000972531.1,PISD,chr22,-,31650699,31648276,31648276,31650699,2423



📊 Intron length statistics:
   Mean: 4433 bp
   Min:  3 bp
   Max:  326803 bp


---
## Extracting Promoter Regions (-1kb to TSS)

### What is a Promoter Region?

```
Promoter = Regulatory region upstream of gene
           Contains transcription factor binding sites

Common definition: -1000 bp to TSS

    -1000bp        TSS      gene
       |------------|=========>
       promoter
```

In [18]:
### Extract Promoter Coordinates


def extract_promoter_regions(tss_df, upstream=1000, downstream=0):
    """
    Extract promoter regions around TSS

    Parameters:
    - upstream: bp upstream of TSS (default 1000)
    - downstream: bp downstream of TSS (default 0)

    Returns DataFrame with promoter coordinates
    """

    promoter_data = []

    for idx, row in tss_df.iterrows():
        tss = row['tss']
        strand = row['strand']

        if strand == '+':
            # For + strand: promoter is upstream (smaller coordinates)
            prom_start = tss - upstream
            prom_end = tss + downstream
        else:
            # For - strand: promoter is upstream (larger coordinates)
            prom_start = tss - downstream
            prom_end = tss + upstream

        # Ensure start < end
        prom_start, prom_end = min(prom_start, prom_end), max(prom_start, prom_end)

        # Don't allow negative coordinates
        prom_start = max(1, prom_start)

        promoter_data.append({
            'gene_name': row['gene_name'],
            'transcript_id': row['transcript_id'],
            'chromosome': row['chromosome'],
            'strand': row['strand'],
            'tss': tss,
            'promoter_start': prom_start,
            'promoter_end': prom_end,
            'promoter_length': prom_end - prom_start
        })

    return pd.DataFrame(promoter_data)

# Extract promoter regions (-1kb to TSS)
promoter_df = extract_promoter_regions(tss_df, upstream=1000, downstream=0)

print("✅ Extracted promoter regions!")
print(f"\nPromoter definition: -1000 bp to TSS")
print(f"\nPromoter data:")
display(promoter_df)

✅ Extracted promoter regions!

Promoter definition: -1000 bp to TSS

Promoter data:


,gene_name,transcript_id,chromosome,strand,tss,promoter_start,promoter_end,promoter_length
0,ENSG00000294541,ENST00000724296.1,chr22,-,10529164,10529164,10530164,1000
1,U2,ENST00000615943.1,chr22,-,10736283,10736283,10737283,1000
2,ENSG00000301473,ENST00000779064.1,chr22,+,10742050,10741050,10742050,1000
3,ENSG00000301473,ENST00000779065.1,chr22,+,10742092,10741092,10742092,1000
4,ENSG00000301496,ENST00000779294.1,chr22,-,10754413,10754413,10755413,1000
...,...,...,...,...,...,...,...,...
11610,RABL2B,ENST00000468451.1,chr22,-,50783600,50783600,50784600,1000
11611,RABL2B,ENST00000464740.1,chr22,-,50783630,50783630,50784630,1000
11612,RABL2B,ENST00000413505.1,chr22,-,50783045,50783045,50784045,1000
11613,ENSG00000307555,ENST00000827060.1,chr22,-,50798991,50798991,50799991,1000


In [19]:
# Save promoter coordinates as BED file
bed_file = 'promoter_regions.bed'
with open(bed_file, 'w') as f:
    for idx, row in promoter_df.iterrows():
        f.write(f"{row['chromosome']}\t{row['promoter_start']}\t{row['promoter_end']}\t")
        f.write(f"{row['gene_name']}\t.\t{row['strand']}\n")

print(f"\n✅ Saved promoter coordinates: {bed_file}")
print("   (BED format - can be used with bedtools or genome browsers)")


✅ Saved promoter coordinates: promoter_regions.bed
   (BED format - can be used with bedtools or genome browsers)


---

## Getting FASTA Sequences

### Understanding FASTA Format

```
FASTA format:
>header (starts with >)
SEQUENCE
SEQUENCE
...

Example:
>chr22:10735171-10736171 RBFOX2 promoter
ATGCGATCGATCGATCG...
```

In [22]:

# This is how you'd extract real sequences from genome FASTA

"""
# First, download genome FASTA (one-time, ~1GB per chromosome)
# Example for chromosome 22:
!wget http://ftp.ensembl.org/pub/release-110/fasta/homo_sapiens/dna/Homo_sapiens.GRCh38.dna.chromosome.22.fa.gz
!gunzip Homo_sapiens.GRCh38.dna.chromosome.22.fa.gz
"""
# Then use pyfaidx:
!pip install pyfaidx


In [25]:
# Extract FASTA sequences using pyfaidx

from pyfaidx import Fasta

# Download genome FASTA for chromosome 22 (if not already downloaded)
# URL: http://ftp.ensembl.org/pub/release-110/fasta/homo_sapiens/dna/Homo_sapiens.GRCh38.dna.chromosome.22.fa.gz

print("📥 Downloading chromosome 22 FASTA...")
import urllib.request
genome_url = "http://ftp.ensembl.org/pub/release-110/fasta/homo_sapiens/dna/Homo_sapiens.GRCh38.dna.chromosome.22.fa.gz"
urllib.request.urlretrieve(genome_url, "Homo_sapiens.GRCh38.dna.chromosome.22.fa.gz")

# Decompress
import gzip
import shutil
with gzip.open('Homo_sapiens.GRCh38.dna.chromosome.22.fa.gz', 'rb') as f_in:
    with open('Homo_sapiens.GRCh38.dna.chromosome.22.fa', 'wb') as f_out:
        shutil.copyfileobj(f_in, f_out)

print("✅ Downloaded and extracted!")

# Load genome with pyfaidx
genome = Fasta('Homo_sapiens.GRCh38.dna.chromosome.22.fa')

print(f"\n📖 Genome loaded!")
print(f"   Chromosomes: {list(genome.keys())}")

def reverse_complement(seq):
    """Reverse complement a DNA sequence"""
    complement = {'A': 'T', 'T': 'A', 'G': 'C', 'C': 'G', 'N': 'N'}
    return ''.join(complement.get(base.upper(), base) for base in reversed(seq))

def extract_sequence(chrom, start, end, strand):
    """
    Extract sequence from genome

    Parameters:
    - chrom: Chromosome name (e.g., '22' or 'chr22')
    - start: Start position (0-based)
    - end: End position (exclusive)
    - strand: '+' or '-'
    """
    # pyfaidx uses 0-based indexing (like Python)
    seq = str(genome[chrom][start:end])

    # Reverse complement if minus strand
    if strand == '-':
        seq = reverse_complement(seq)

    return seq

📥 Downloading chromosome 22 FASTA...
✅ Downloaded and extracted!

📖 Genome loaded!
   Chromosomes: ['22']


In [41]:
# Example 1: Extract genome sequence of chromosome 22 (plus strand)
seq1 = extract_sequence('22', 17662484, 17663484, '+')
print(f"   Chr22:17662484-17663484 (+)")
print(f"   Length: {len(seq1)} bp")
print(f"   Sequence: {seq1[:80]}...")

   Chr22:17662484-17663484 (+)
   Length: 1000 bp
   Sequence: AAACAAAAAATTTACTGAAACCTGAATTAGTCTTCTTCATTAAAATATAACGTCAGGCTGGGTGTCATGGCTCACACCTG...


In [43]:
# Example 2: Extract from minus strand
print("\n🧬 Example 2: Minus strand (with reverse complement)")
seq2 = extract_sequence('22', 17662484, 17663484, '-')
print(f"   Chr22:17662484-17663484 (-)")
print(f"   Length: {len(seq2)} bp")
print(f"   Sequence: {seq2[:80]}...")


🧬 Example 2: Minus strand (with reverse complement)
   Chr22:17662484-17663484 (-)
   Length: 1000 bp
   Sequence: TTTGATGAGATTATAAATATTTGGTATCGCATGATGTTTTATGACCTTTTAGTCATTTTGATATAATTTTGTTTAGATAG...


In [37]:
# Example 3: Extract promoter region for a gene
print("\n🧬 Example 3: Extract promoter region")
# Example: RBFOX2 on chr22, TSS at position 10755301, minus strand
tss = 10755301
promoter_start = tss  # For minus strand
promoter_end = tss + 1000
promoter_seq = extract_sequence('22', promoter_start, promoter_end, '-')
print(f"   RBFOX2 promoter (chr22:{promoter_start}-{promoter_end}, -)")
print(f"   Length: {len(promoter_seq)} bp")
print(f"   Sequence: {promoter_seq[:80]}...")


🧬 Example 3: Extract promoter region
   RBFOX2 promoter (chr22:10755301-10756301, -)
   Length: 1000 bp
   Sequence: AGATTGGTGAAGATTAAAAACAAATTTGGAGCATGGGGAGCCTTACAATACTTATTTATTAGAGCACGTAGCCTAAAACC...


In [40]:
examples = [
    # Gene regions from your GTF (these have real sequence)
    ('RBFOX2 promoter', '22', 10755301, 10756301, '-'),
    ('PTPN1 gene start', '22', 17662484, 17663484, '+'),
    ('CPT1B gene start', '22', 23523149, 23524149, '+'),
]

for name, chrom, start, end, strand in examples:
    seq = str(genome[chrom][start:end])

    if strand == '-':
        complement = {'A': 'T', 'T': 'A', 'G': 'C', 'C': 'G', 'N': 'N'}
        seq = ''.join(complement.get(b.upper(), b) for b in reversed(seq))

    n_count = seq.count('N') + seq.count('n')

    print(f"\n{name}")
    print(f"  Location: chr{chrom}:{start}-{end} ({strand})")
    print(f"  Length: {len(seq)} bp")
    print(f"  N content: {n_count} bp ({n_count/len(seq)*100:.1f}%)")
    print(f"  Sequence: {seq[:80]}...")


RBFOX2 promoter
  Location: chr22:10755301-10756301 (-)
  Length: 1000 bp
  N content: 0 bp (0.0%)
  Sequence: AGATTGGTGAAGATTAAAAACAAATTTGGAGCATGGGGAGCCTTACAATACTTATTTATTAGAGCACGTAGCCTAAAACC...

PTPN1 gene start
  Location: chr22:17662484-17663484 (+)
  Length: 1000 bp
  N content: 0 bp (0.0%)
  Sequence: AAACAAAAAATTTACTGAAACCTGAATTAGTCTTCTTCATTAAAATATAACGTCAGGCTGGGTGTCATGGCTCACACCTG...

CPT1B gene start
  Location: chr22:23523149-23524149 (+)
  Length: 1000 bp
  N content: 0 bp (0.0%)
  Sequence: GGATTCGGGGGCTTGTGGGGAGGTCTCCACTGGGATCAGATGGGCCTGAAGGACGCCCCCCACCCTCAAGCCCCTGCTGG...


In [44]:
# ============================================================================
# Part 9: Regular Expressions and Motif Finding
# ============================================================================

print("="*70)
print("PART 9: REGULAR EXPRESSIONS AND MOTIF FINDING")
print("="*70)

# ----------------------------------------------------------------------------
# 9.1: What is a Regular Expression?
# ----------------------------------------------------------------------------

print("\n" + "="*70)
print("9.1: What is a Regular Expression (Regex)?")
print("="*70)

print("""
Regular Expression = A pattern matching language

Think of it as a "search pattern" for text:
- Find specific sequences
- Allow variations
- Match multiple possibilities

Like Google search, but for DNA!
""")


PART 9: REGULAR EXPRESSIONS AND MOTIF FINDING

9.1: What is a Regular Expression (Regex)?

Regular Expression = A pattern matching language

Think of it as a "search pattern" for text:
- Find specific sequences
- Allow variations
- Match multiple possibilities

Like Google search, but for DNA!



In [45]:
# Basic regex example with normal text
import re

print("📝 Simple Text Example:\n")

text = "My phone number is 555-1234 and my friend's is 555-9876"

# Pattern: Find phone numbers
pattern = r'\d{3}-\d{4}'  # \d = digit, {3} = exactly 3 times

matches = re.findall(pattern, text)
print(f"Text: {text}")
print(f"Pattern: {pattern}")
print(f"Found: {matches}")
print("Explanation: \\d{{3}}-\\d{{4}} means '3 digits, dash, 4 digits'\n")

📝 Simple Text Example:

Text: My phone number is 555-1234 and my friend's is 555-9876
Pattern: \d{3}-\d{4}
Found: ['555-1234', '555-9876']
Explanation: \d{{3}}-\d{{4}} means '3 digits, dash, 4 digits'



In [46]:
# ----------------------------------------------------------------------------
# 9.2: Regular Expression Basics
# ----------------------------------------------------------------------------

print("="*70)
print("9.2: Regex Basics - Common Patterns")
print("="*70)

examples = [
    (".",        "Any single character",              "A.C matches ABC, ACC, ATC"),
    ("[ATGC]",   "Any one of these characters",       "[AT] matches A or T"),
    ("[A-Z]",    "Range of characters",               "[A-Z] matches any uppercase letter"),
    ("*",        "Zero or more times",                "A* matches '', A, AA, AAA..."),
    ("+",        "One or more times",                 "A+ matches A, AA, AAA..."),
    ("?",        "Zero or one time",                  "A? matches '' or A"),
    ("{n}",      "Exactly n times",                   "A{3} matches AAA"),
    ("{n,m}",    "Between n and m times",             "A{2,4} matches AA, AAA, AAAA"),
]

print("\nCommon Regex Symbols:\n")
for symbol, meaning, example in examples:
    print(f"{symbol:12s} → {meaning:30s} → {example}")


9.2: Regex Basics - Common Patterns

Common Regex Symbols:

.            → Any single character           → A.C matches ABC, ACC, ATC
[ATGC]       → Any one of these characters    → [AT] matches A or T
[A-Z]        → Range of characters            → [A-Z] matches any uppercase letter
*            → Zero or more times             → A* matches '', A, AA, AAA...
+            → One or more times              → A+ matches A, AA, AAA...
?            → Zero or one time               → A? matches '' or A
{n}          → Exactly n times                → A{3} matches AAA
{n,m}        → Between n and m times          → A{2,4} matches AA, AAA, AAAA


In [47]:
# ----------------------------------------------------------------------------
# 9.3: DNA Sequence Regex Example
# ----------------------------------------------------------------------------

print("\n" + "="*70)
print("9.3: DNA Sequence Example")
print("="*70)

# Sample DNA sequence
dna_sequence = "ATGCGATCGATCGTAGCTAGCTAGATGCCCGTAATGCCC"

print(f"\nDNA Sequence:\n{dna_sequence}\n")

# Example 1: Find ATG (start codon)
print("Example 1: Find all ATG (start codons)")
pattern1 = "ATG"
matches1 = [(m.start(), m.group()) for m in re.finditer(pattern1, dna_sequence)]
print(f"Pattern: {pattern1}")
print(f"Matches: {len(matches1)}")
for pos, seq in matches1:
    print(f"  Position {pos}: {seq}")

# Example 2: Find any sequence starting with A and ending with G
print("\nExample 2: Find A followed by 2 bases then G")
pattern2 = "A..G"  # A, any char, any char, G
matches2 = [(m.start(), m.group()) for m in re.finditer(pattern2, dna_sequence)]
print(f"Pattern: {pattern2}")
print(f"Matches: {len(matches2)}")
for pos, seq in matches2:
    print(f"  Position {pos}: {seq}")

# Example 3: Find purines (A or G) repeated 3 times
print("\nExample 3: Find 3 consecutive purines (A or G)")
pattern3 = "[AG]{3}"
matches3 = [(m.start(), m.group()) for m in re.finditer(pattern3, dna_sequence)]
print(f"Pattern: {pattern3}")
print(f"Explanation: [AG] means A or G, {{3}} means exactly 3 times")
print(f"Matches: {len(matches3)}")
for pos, seq in matches3:
    print(f"  Position {pos}: {seq}")


9.3: DNA Sequence Example

DNA Sequence:
ATGCGATCGATCGTAGCTAGCTAGATGCCCGTAATGCCC

Example 1: Find all ATG (start codons)
Pattern: ATG
Matches: 3
  Position 0: ATG
  Position 24: ATG
  Position 33: ATG

Example 2: Find A followed by 2 bases then G
Pattern: A..G
Matches: 3
  Position 5: ATCG
  Position 9: ATCG
  Position 32: AATG

Example 3: Find 3 consecutive purines (A or G)
Pattern: [AG]{3}
Explanation: [AG] means A or G, {3} means exactly 3 times
Matches: 1
  Position 22: AGA


In [48]:
# ----------------------------------------------------------------------------
# 9.4: What is a DNA Motif?
# ----------------------------------------------------------------------------

print("\n" + "="*70)
print("9.4: What is a DNA Motif?")
print("="*70)

print("""
DNA Motif = A recurring pattern in DNA sequences

Common types:
1. Transcription Factor Binding Sites
   - Proteins bind to specific DNA patterns
   - Regulate gene expression

2. Promoter elements
   - TATA box: TATAAA
   - CAAT box: GGCCAATCT

3. Splice sites
   - Donor site: GT
   - Acceptor site: AG

Example: TATA Box
   Pattern: TATAAA (or similar)
   Location: ~30 bp upstream of gene start
   Function: RNA polymerase binding
""")

# Find TATA box example
print("🔍 Finding TATA box:\n")

promoter_seq = "GCGCGCTATAAAAGGCTAGCTAGCTAGC"
tata_pattern = "TATA[AT]A"  # TATA + (A or T) + A

print(f"Promoter sequence: {promoter_seq}")
print(f"Pattern: {tata_pattern}")
print(f"Explanation: TATA + [AT] + A means TATA, then A or T, then A")

match = re.search(tata_pattern, promoter_seq)
if match:
    print(f"\n✅ Found TATA box!")
    print(f"   Position: {match.start()}")
    print(f"   Sequence: {match.group()}")
else:
    print("\n❌ No TATA box found")


9.4: What is a DNA Motif?

DNA Motif = A recurring pattern in DNA sequences

Common types:
1. Transcription Factor Binding Sites
   - Proteins bind to specific DNA patterns
   - Regulate gene expression
   
2. Promoter elements
   - TATA box: TATAAA
   - CAAT box: GGCCAATCT
   
3. Splice sites
   - Donor site: GT
   - Acceptor site: AG

Example: TATA Box
   Pattern: TATAAA (or similar)
   Location: ~30 bp upstream of gene start
   Function: RNA polymerase binding

🔍 Finding TATA box:

Promoter sequence: GCGCGCTATAAAAGGCTAGCTAGCTAGC
Pattern: TATA[AT]A
Explanation: TATA + [AT] + A means TATA, then A or T, then A

✅ Found TATA box!
   Position: 6
   Sequence: TATAAA


In [51]:
# ----------------------------------------------------------------------------
# 9.5: IUPAC(International Union of Pure and Applied Chemistry) Nucleotide Codes (for motifs)
# ----------------------------------------------------------------------------

print("\n" + "="*70)
print("9.5: IUPAC(International Union of Pure and Applied Chemistry) Nucleotide Codes (DNA Ambiguity)")
print("="*70)

iupac_codes = {
    'A': 'Adenine',
    'T': 'Thymine',
    'G': 'Guanine',
    'C': 'Cytosine',
    'R': 'Purine (A or G)',
    'Y': 'Pyrimidine (C or T)',
    'W': 'Weak bond (A or T)',
    'S': 'Strong bond (G or C)',
    'M': 'aMino (A or C)',
    'K': 'Keto (G or T)',
    'N': 'aNy (A, T, G, or C)',
}

print("\nCommon IUPAC codes used in motifs:\n")
for code, meaning in list(iupac_codes.items())[:11]:
    print(f"  {code} = {meaning}")

print("\n💡 These codes describe flexible motif patterns!")
print("   Example: RYN means (A or G)(C or T)(any base)")

# Convert IUPAC to regex
print("\n🔧 Converting IUPAC to Regex:\n")

iupac_to_regex = {
    'R': '[AG]',    # Purine
    'Y': '[CT]',    # Pyrimidine
    'W': '[AT]',    # Weak
    'S': '[GC]',    # Strong
    'M': '[AC]',    # aMino
    'K': '[GT]',    # Keto
    'N': '[ATGC]',  # aNy
}

for iupac, regex in iupac_to_regex.items():
    print(f"  {iupac} → {regex}")


9.5: IUPAC(International Union of Pure and Applied Chemistry) Nucleotide Codes (DNA Ambiguity)

Common IUPAC codes used in motifs:

  A = Adenine
  T = Thymine
  G = Guanine
  C = Cytosine
  R = Purine (A or G)
  Y = Pyrimidine (C or T)
  W = Weak bond (A or T)
  S = Strong bond (G or C)
  M = aMino (A or C)
  K = Keto (G or T)
  N = aNy (A, T, G, or C)

💡 These codes describe flexible motif patterns!
   Example: RYN means (A or G)(C or T)(any base)

🔧 Converting IUPAC to Regex:

  R → [AG]
  Y → [CT]
  W → [AT]
  S → [GC]
  M → [AC]
  K → [GT]
  N → [ATGC]
